# Práctica: dos variables y discretización

**MA2015.201**

Completamos una generación de un algoritmo genético para encontrar una posición cercana a $(1.3,2.7)$ dentro del cuadrado de 0 a 4 unidades por eje. Evaluamos la población inicial, seleccionamos padres, cruzamos sus genes, aplicamos mutaciones y conservamos al mejor observado. Al terminar, obtenemos las tablas, los archivos y la gráfica del ejemplo.

Completa los **13 bloques `TODO`**. Sustituye cada `raise NotImplementedError(...)` por tu implementación; en las funciones incluye el retorno y en las actividades de integración construye las variables indicadas. Usa las fórmulas y las instrucciones de cada sección. Reinicia el kernel y ejecuta las celdas en orden.

Los datos de población, torneos y mutaciones están fijados para reproducir la misma generación. Usamos Python, NumPy y Matplotlib. Conserva precisión completa en los cálculos; el formato de salida muestra dos decimales. Al final encontrarás los resultados de referencia para contrastar tu trabajo.


## 1. Dos decisiones y una aptitud

Elegimos una posición $(x,y)$: $x$ representa la decisión horizontal y $y$ la vertical. Medimos su cercanía a la meta mediante la distancia al cuadrado:

$$F(x,y)=(x-1.3)^2+(y-2.7)^2.$$

Una aptitud menor representa mayor cercanía. En la versión continua, la meta pertenece al dominio y tiene aptitud cero: es el minimizador porque la suma de cuadrados es no negativa.

En la representación discreta usamos paso $0.5$ y genes enteros $k_1,k_2\in\{0,1,\ldots,8\}$:

$$x=0.5k_1,\qquad y=0.5k_2.$$

Cada gen codifica una decisión. El cromosoma contiene ambos genes; un individuo representa un candidato y una población reúne varios individuos. La rejilla admite $9\times9=81$ cromosomas.

Ejecuta la configuración y la función de presentación. Estos bloques están completos.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

TARGET = np.array([1.3, 2.7], dtype=float)
UPPER = 4.0
STEP = 0.5
MAX_LEVEL = 8


def show_population(population: dict, metrics: dict) -> None:
    """Presentación de cromosomas, coordenadas y aptitudes ya calculadas."""
    print(f"{'ID':<3} {'Genes':<10} {'Coordenadas':<14} {'Aptitud':>8}")
    for name, genes in population.items():
        chromosome = tuple(int(value) for value in genes)
        coordinates = tuple(float(value) for value in metrics[name]["coordinates"])
        fitness = metrics[name]["fitness"]
        print(f"{name:<3} {str(chromosome):<10} {str(coordinates):<14} {fitness:>8.2f}")


### TODO 1 · Calcula la aptitud

Completa `squared_distance`. Recibe dos arreglos de dos coordenadas, `position` y `target`, y devuelve un `float` con la suma de los cuadrados de sus diferencias. Usa la meta recibida como parámetro.

Calcula a mano las aptitudes de $(1,2)$, $(2,2.5)$ y la meta; después ejecuta la celda de salida.


In [ ]:
def squared_distance(position: np.ndarray, target: np.ndarray) -> float:
    """Distancia al cuadrado entre una posición y una meta."""
    # TODO 1: cálculo de las diferencias, suma de cuadrados y retorno.
    raise NotImplementedError("TODO 1: squared_distance")


In [ ]:
for point in [np.array([1.0, 2.0]), np.array([2.0, 2.5]), TARGET]:
    print(point, round(squared_distance(point, TARGET), 2))


## 2. Decodifica y evalúa un cromosoma

**TODO 2.** Completa `decode`: recibe dos genes enteros y devuelve un arreglo de coordenadas aplicando el paso recibido. `max_level` indica el nivel máximo permitido; los datos proporcionados pertenecen a ese rango. Conserva el arreglo de entrada.

**TODO 3.** Completa `evaluate`: decodifica los genes, calcula la aptitud respecto de `target` y devuelve un diccionario con dos claves:

- `coordinates`: las coordenadas obtenidas como arreglo.
- `fitness`: su aptitud con precisión completa.

Reutiliza `decode` y `squared_distance` en el evaluador. Ejecuta la referencia de rejilla después de completar las funciones.

La referencia de rejilla elige la coordenada permitida más cercana a la meta en cada eje. Como sumamos dos cuadrados separados, podemos minimizar cada diferencia por separado.


In [ ]:
def decode(genes: np.ndarray, step: float, max_level: int) -> np.ndarray:
    """Conversión de dos niveles enteros a coordenadas."""
    # TODO 2: conversión de genes a coordenadas, sin modificar la entrada.
    raise NotImplementedError("TODO 2: decode")


In [ ]:
def evaluate(
    genes: np.ndarray, step: float, target: np.ndarray, max_level: int
) -> dict:
    """Coordenadas y aptitud de un cromosoma."""
    # TODO 3: composición de las funciones y construcción del diccionario.
    raise NotImplementedError("TODO 3: evaluate")


In [ ]:
reference = evaluate(np.array([3, 5]), STEP, TARGET, MAX_LEVEL)
print("Referencia de rejilla:", reference["coordinates"], round(reference["fitness"], 2))


## 3. Evalúa la población inicial y guarda su campeón

En `P0`, cada etiqueta identifica un individuo y cada arreglo contiene su cromosoma completo. Los cuatro candidatos son datos del ejemplo.

**TODO 4.** Recorre `P0` y construye `initial_metrics`. Conserva las mismas etiquetas y guarda para cada una el diccionario devuelto por `evaluate`.

**TODO 5.** Identifica al individuo de menor aptitud y construye `archive0`, el registro inicial del mejor observado. Usa esta estructura:

- `genes`: copia de su cromosoma.
- `metrics`: diccionario con copia de sus coordenadas y su aptitud.

Usa `.copy()` en los arreglos para que el archivo conserve un registro independiente. Obtén al campeón mediante las aptitudes calculadas, sin fijar su etiqueta en el código.


In [ ]:
P0 = {
    "A": np.array([2, 4]),
    "B": np.array([6, 2]),
    "C": np.array([4, 5]),
    "D": np.array([1, 7]),
}


In [ ]:
initial_metrics = {}
# TODO 4: evaluación de cada candidato y almacenamiento por etiqueta.
raise NotImplementedError("TODO 4: initial_metrics")

show_population(P0, initial_metrics)


In [ ]:
# TODO 5: selección del campeón y construcción independiente de archive0.
raise NotImplementedError("TODO 5: archive0")

print("Archivo inicial:", archive0)


## 4. Selecciona padres, cruza y muta

Todos los padres de esta generación pertenecen a `P0`.

**TODO 6 · Torneo.** Completa `tournament_winner`: recibe las dos etiquetas de un torneo y las métricas de la población. Devuelve la etiqueta de menor aptitud; en un empate conserva al primer participante.

**TODO 7 · Cruza.** Completa `crossover`: forma un cromosoma nuevo con el primer gen de `parent1` y el segundo de `parent2`.

**TODO 8 · Mutación.** Completa `mutate`: suma `increments` a los genes y recorta cada resultado al intervalo de 0 a `max_level`. Cada incremento cambia un nivel, equivalente a $0.5$ unidades en su eje. Puedes usar `np.clip` para el recorte.

Las funciones de cruza y mutación devuelven arreglos nuevos y conservan sus entradas.


In [ ]:
def tournament_winner(participants: tuple[str, str], metrics: dict) -> str:
    """Etiqueta ganadora de un torneo de dos candidatos."""
    # TODO 6: comparación de aptitudes y retorno de la etiqueta ganadora.
    raise NotImplementedError("TODO 6: tournament_winner")


In [ ]:
def crossover(parent1: np.ndarray, parent2: np.ndarray) -> np.ndarray:
    """Cromosoma nuevo formado con un gen de cada padre."""
    # TODO 7: primer gen del primer padre y segundo gen del segundo padre.
    raise NotImplementedError("TODO 7: crossover")


In [ ]:
def mutate(genes: np.ndarray, increments: np.ndarray, max_level: int) -> np.ndarray:
    """Aplicación de incrementos y recorte al dominio de niveles."""
    # TODO 8: suma de incrementos y recorte de los genes al dominio.
    raise NotImplementedError("TODO 8: mutate")


## 5. Construye los cuatro hijos

Fijamos los participantes de los torneos y los incrementos para reproducir la generación:

| Hijo | Torneo del primer padre | Torneo del segundo padre | Incrementos de mutación |
|---|---|---|---|
| E | A / B | C / D | $(0,+1)$ |
| F | C / B | A / D | $(+1,0)$ |
| G | C / B | D / B | $(0,-1)$ |
| H | D / B | C / A | $(0,-1)$ |

### Ejemplo manual: E

Seleccionamos A en el primer torneo y C en el segundo. Sus cromosomas son $(2,4)$ y $(4,5)$; la cruza produce $(2,5)$. Aplicamos $(0,+1)$ y obtenemos $(2,6)$, que representa $(1,3)$:

$$F(1,3)=(-0.3)^2+(0.3)^2=0.18.$$

### TODO 9 · Integra los operadores

Construye E, F, G y H mediante tus funciones. Recorre `CHILD_PLAN` y, para cada hijo:

1. Obtén las dos etiquetas ganadoras de los torneos usando `initial_metrics`.
2. Recupera ambos cromosomas de `P0` y crúzalos.
3. Aplica los incrementos indicados y guarda el cromosoma final en `CHILDREN`.
4. Evalúa ese cromosoma y guarda el resultado en `child_metrics`.
5. Guarda en `generation_trace` un diccionario con `parents` (par de etiquetas seleccionadas) y `before_mutation` (copia del cromosoma cruzado).

Deriva primero F, G y H a mano y contrasta tus cálculos con la traza impresa. Calcula los hijos con los operadores; conserva los datos de torneos y mutaciones tal como se proporcionan.


In [ ]:
CHILD_PLAN = {
    "E": {"tournaments": (("A", "B"), ("C", "D")), "increments": np.array([0, 1])},
    "F": {"tournaments": (("C", "B"), ("A", "D")), "increments": np.array([1, 0])},
    "G": {"tournaments": (("C", "B"), ("D", "B")), "increments": np.array([0, -1])},
    "H": {"tournaments": (("D", "B"), ("C", "A")), "increments": np.array([0, -1])},
}


In [ ]:
CHILDREN = {}
child_metrics = {}
generation_trace = {}
# TODO 9: construcción de cada hijo y almacenamiento de su traza y métricas.
raise NotImplementedError("TODO 9: generación de E, F, G y H")

for name, trace in generation_trace.items():
    print(name, "padres:", trace["parents"], "antes de mutar:", trace["before_mutation"],
          "incrementos:", CHILD_PLAN[name]["increments"], "final:", CHILDREN[name])
show_population(CHILDREN, child_metrics)


## 6. Conserva al campeón y actualiza el archivo

La población siguiente contiene cuatro individuos. El **reemplazo elitista** compara al campeón de `P0` con el peor hijo: lo conserva en la nueva población sólo cuando su aptitud es estrictamente menor que la del peor hijo.

**TODO 10.** Completa `elitist_replacement`. Identifica ambas etiquetas mediante las métricas y construye una copia de la población de hijos. Cuando corresponda, sustituye al peor por el campeón inicial en la misma posición del recorrido del diccionario, usando la etiqueta del campeón. Conserva copias independientes de los cromosomas. Si no hay mejora estricta, conserva los cuatro hijos. Ante empates, utiliza la primera etiqueta encontrada.

El **archivo** tiene otra función: conserva al mejor observado a lo largo de la búsqueda, aunque cambie la población.

**TODO 11.** Completa `update_archive`. Identifica al mejor candidato de la población recibida y compáralo con `archive`. Si mejora estrictamente su aptitud, devuelve su registro; de lo contrario, conserva el anterior. En ambos casos devuelve copias de genes y coordenadas. Mantén la estructura `genes` y `metrics` utilizada en `archive0`.

La referencia de rejilla sirve para interpretar el resultado; el archivo se actualiza con los candidatos de la población.


In [ ]:
def elitist_replacement(
    P0: dict, children: dict, initial_metrics: dict, child_metrics: dict
) -> dict:
    """Nueva población tras comparar al campeón inicial con el peor hijo."""
    # TODO 10: selección, comparación y copia de la población con reemplazo si corresponde.
    raise NotImplementedError("TODO 10: elitist_replacement")


In [ ]:
def update_archive(archive: dict, population: dict, metrics: dict) -> dict:
    """Registro independiente del mejor candidato observado."""
    # TODO 11: comparación con el archivo y copia del registro que se conserva.
    raise NotImplementedError("TODO 11: update_archive")


### TODO 12 · Forma la población siguiente y su archivo

Utiliza las funciones anteriores para construir:

- `P1`: resultado del reemplazo elitista aplicado a `P0` y `CHILDREN`.
- `next_metrics`: evaluación de cada individuo de `P1`.
- `archive1`: actualización de `archive0` a partir de `P1` y sus métricas.

Obtén los resultados mediante las funciones, sin transcribir etiquetas ganadoras o cromosomas finales. Conserva el orden de la población. Ejecuta la salida y explica qué operación afecta a la población y cuál afecta al archivo.


In [ ]:
# TODO 12: construcción de P1, next_metrics y archive1 mediante las funciones.
raise NotImplementedError("TODO 12: población siguiente y archivo")

print("Población siguiente:", tuple(P1))
show_population(P1, next_metrics)
print("Archivo inicial:", archive0)
print("Archivo final:", archive1)


## 7. Interpreta la rejilla y el resultado

Ejecuta la gráfica proporcionada cuando hayas completado la generación. Comparamos tres referentes:

- La meta continua, cuyo valor mínimo es cero.
- La mejor posición permitida por la rejilla.
- El mejor candidato observado y conservado en `archive1`.

Explica qué diferencia se debe a la discretización y qué diferencia permanece por la búsqueda realizada. Obtener al mejor de una generación no demuestra por sí mismo optimalidad.


In [ ]:
levels = np.arange(MAX_LEVEL + 1)
grid_x, grid_y = np.meshgrid(levels * STEP, levels * STEP)
fig, ax = plt.subplots(figsize=(6.4, 6.4))
ax.scatter(grid_x, grid_y, s=12, color="0.72")
ax.scatter(*TARGET, marker="*", s=180, color="#b42318", label='Meta continua')
ax.scatter(*reference["coordinates"], marker="s", s=70, color="#003670",
           label='Mejor permitido')
ax.scatter(*archive1["metrics"]["coordinates"], marker="^", s=80,
           color="#167044", label='Mejor observado E')
ax.set(xlim=(-0.15, UPPER + 0.15), ylim=(-0.15, UPPER + 0.15),
       xlabel='Posición horizontal x (unidades)', ylabel='Posición vertical y (unidades)')
ax.set_aspect("equal")
ax.legend(loc="lower right")
fig.tight_layout()
plt.show()


## 8. Aplica la representación a otra posición

**TODO 13.** Codifica la posición $(2.5,1.5)$, aumenta un nivel vertical y evalúa el resultado usando tus funciones.

Construye `exercise_genes`, `exercise_mutated` y `exercise_result`. Obtén los genes a partir de la relación entre coordenadas y paso; aplica la mutación $(0,+1)$ y utiliza `evaluate` para calcular posición y aptitud.


In [ ]:
exercise_position = np.array([2.5, 1.5])
exercise_increments = np.array([0, 1])
# TODO 13: codificación, mutación y evaluación de la nueva posición.
raise NotImplementedError("TODO 13: nueva posición")

print("Genes iniciales:", exercise_genes)
print("Genes mutados:", exercise_mutated)
print("Posición:", exercise_result["coordinates"])
print("Aptitud:", round(exercise_result["fitness"], 2))


## Resultados de referencia

Compara tus salidas con estos valores después de completar las actividades.

Los tres ejemplos de aptitud producen **0.58, 0.53 y 0.00**. La referencia de rejilla $(3,5)$ representa $(1.5,2.5)$ y tiene aptitud **0.08**.

### Población inicial

| ID | Cromosoma | Posición | Aptitud |
|---|---|---|---:|
| A | $(2,4)$ | $(1,2)$ | $0.58$ |
| B | $(6,2)$ | $(3,1)$ | $5.78$ |
| C | $(4,5)$ | $(2,2.5)$ | $0.53$ |
| D | $(1,7)$ | $(0.5,3.5)$ | $1.28$ |

### Hijos y recorrido de operadores

| Hijo | Padres | Antes de mutar | Cromosoma final | Posición | Aptitud |
|---|---|---|---|---|---:|
| E | A / C | $(2,5)$ | $(2,6)$ | $(1,3)$ | $0.18$ |
| F | C / A | $(4,4)$ | $(5,4)$ | $(2.5,2)$ | $1.93$ |
| G | C / D | $(4,7)$ | $(4,6)$ | $(2,3)$ | $0.58$ |
| H | D / C | $(1,5)$ | $(1,4)$ | $(0.5,2)$ | $1.13$ |

El campeón inicial es C y el peor hijo es F. El reemplazo conserva la población **E, C, G, H** con aptitudes **0.18, 0.53, 0.58 y 1.13**.

El archivo inicial conserva C, genes $(4,5)$, posición $(2,2.5)$ y aptitud $0.53$. El archivo final conserva E, genes $(2,6)$, posición $(1,3)$ y aptitud $0.18$.

El ejercicio final produce genes iniciales $(5,3)$, genes mutados $(5,4)$, posición $(2.5,2)$ y aptitud **1.93**.


## Explica el recorrido

Conserva tus cálculos y responde:

1. ¿Qué elegimos y qué calculamos al evaluar un candidato?
2. ¿Cómo distinguimos coordenada, nivel, cromosoma y aptitud?
3. ¿Por qué se reemplaza F por C y se actualiza el archivo a E?
4. ¿Por qué conviene copiar los arreglos al formar la población y el archivo?
5. ¿Por qué el mejor observado tiene aptitud $0.18$ y el mejor permitido tiene $0.08$?
6. ¿Qué cambiaría al reducir el paso y conservar el dominio de 0 a 4?

Al trasladar estas ideas a Planta Física identificamos primero las decisiones y sus límites; después relacionamos su representación con la evaluación de candidatos.

**Respuestas:**

Escribe aquí tus operaciones y explicaciones.
